# 2. HeteroData Builder

This notebook assembles the chunked edge files into a single PyTorch Geometric `HeteroData` object.
It loads node counts and edge indices to construct the graph structure.

In [ ]:
# Configuration
import os

# EXTERNAL DRIVE CONFIGURATION
ROOT_DIR = "/Volumes/Backup Plus/Zaman/graph"
CHUNKS_DIR = os.path.join(ROOT_DIR, "processed_data", "chunks")
OUTPUT_FILE = os.path.join(ROOT_DIR, "processed_data", "hetero_graph.pt")

# Node Types and their counts directories (from Step 0 Mapping)
# If you haven't stored explicit counts, you can query LMDB or use the logs from Step 0.
# Here we provide a helper to read from LMDB if needed, or you can hardcode from Step 0 output.
NODE_DIRS = {
    "pekerja": os.path.join(ROOT_DIR, "lmdb_node_mapping", "pekerja.lmdb"),
    "nasabah": os.path.join(ROOT_DIR, "lmdb_node_mapping", "nasabah.lmdb"),
    # Add others if relevant for the graph structure
}

# Edge Types and their chunk directories (from Step 1)
# Key format: (src_node_type, relation_name, dst_node_type)
EDGE_DIRS = {
    ("nasabah", "is_pekerja", "pekerja"): os.path.join(CHUNKS_DIR, "edges_edge_nasabah_is_pekerja"),
    ("nasabah", "has_simp", "simpanan"): os.path.join(CHUNKS_DIR, "edges_edge_nasabah_memiliki_simp"),
    ("nasabah", "has_pinj", "pinjaman"): os.path.join(CHUNKS_DIR, "edges_edge_nasabah_memiliki_pinj"),
    ("simpanan", "simp_debit_transaksi", "transaksi"): os.path.join(CHUNKS_DIR, "edges_edge_simp_debit"),
    ("transaksi", "transaksi_credit_simp", "simpanan"): os.path.join(CHUNKS_DIR, "edges_edge_simp_credit"),
    # Add all other edges you processed
}

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

In [ ]:
# Imports
import torch
from torch_geometric.data import HeteroData
import glob
from tqdm.notebook import tqdm
import lmdb

In [ ]:
def load_num_nodes(lmdb_path):
    """Estimate num nodes from LMDB stats or manual count."""
    if not os.path.exists(lmdb_path):
        return 0
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    with env.begin() as txn:
        # This might be an overestimate if you have holes, but for contiguous 0..N-1 it works.
        # Ideally you saved the count metadata in Step 0.
        stat = env.stat()
        entries = stat['entries']
    return entries

def load_edge_index(chunk_dir):
    """Load all .pt chunks from directory and concatenate."""
    if not os.path.exists(chunk_dir):
        print(f"Warning: Chunk dir not found {chunk_dir}")
        return torch.empty((2, 0), dtype=torch.long)
        
    files = glob.glob(os.path.join(chunk_dir, "*.pt"))
    files.sort() # Ensure consistent order
    
    edges = []
    for f in tqdm(files, desc=f"Loading {os.path.basename(chunk_dir)}"):
        # edges usually don't need weights_only=False security check as they are pure tensors
        # but new pytorch defaults might warn.
        chunk = torch.load(f, weights_only=True)
        edges.append(chunk)
        
    if not edges:
        return torch.empty((2, 0), dtype=torch.long)
        
    return torch.cat(edges, dim=1)

In [ ]:
def build_hetero_graph(node_dirs, edge_dirs, out_path):
    data = HeteroData()

    # 1. Set Num Nodes
    print("Setting Node Counts...")
    for node_type, path in node_dirs.items():
        count = load_num_nodes(path)
        data[node_type].num_nodes = count
        print(f"  {node_type}: {count}")

    # 2. Load Edges
    print("\nLoading Edges...")
    for edge_type, path in edge_dirs.items():
        print(f"  Processing {edge_type}...")
        edge_index = load_edge_index(path)
        data[edge_type].edge_index = edge_index
        print(f"    Loaded {edge_index.size(1)} edges.")

    # 3. Save
    print(f"\nSaving to {out_path}...")
    torch.save(data, out_path)
    print("Done.")

build_hetero_graph(NODE_DIRS, EDGE_DIRS, OUTPUT_FILE)